In [ ]:
!pip install pycox
!pip install torchtuples
!pip install scikit-survival
!pip install lifelines

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.2/96.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.3/141.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.7/413.7 kB 13.1 MB/s eta 0:00:00
  Created wheel for feather-format: filename=feather_format-0.4.1-py3-none-any.whl size=2435 sha256=213a6efafaf6691f35ee0a85b3429915e7c64afef7e19e74338b7500008f696f
  Stored in directory: /root/.cache/pip/wheels/77/5b/0e/0e63d10b6353208a085a321ea2eed2578

In [ ]:
import pandas as pd
import numpy as np
import torch
import torchtuples as tt
from pycox.models import CoxPH
from lifelines.utils import concordance_index as concordance_td
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sksurv.util import Surv
from pycox.models.loss import CoxPHLoss
from pycox.models import DeepHitSingle
from pycox.models.loss import DeepHitLoss
from torchtuples.practical import MLPVanilla

Load Data

In [ ]:
train_file_path = "1yVTNHjwM4hUbv0alTfW9hEn1itOVbHLg"
test_file_path = "1-1NGzpvsdLcBPFZU_aNOZYn-1BIMoWO2"
train_url = f"https://drive.google.com/uc?export=download&id={train_file_path}"
test_url = f"https://drive.google.com/uc?export=download&id={test_file_path}"
train_df = pd.read_csv(train_url)
test_df = pd.read_csv(test_url)

Preprocess Data

In [ ]:
class DeepSurvModel(torch.nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = torch.nn.Linear(input_dim, 128)
        self.fc2 = torch.nn.Linear(128, 64)
        self.fc3 = torch.nn.Linear(64, 1)  # Output risk score

        self.activation = torch.nn.SiLU()  # Instead of ReLU

    def forward(self, x):
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        x = self.fc3(x)  # No activation for Cox output
        return x



# Initialize DeepSurv Model
# input_dim = train_x.shape[1]
# model = CoxPH(
#     net=DeepSurvModel(input_dim),
#     optimizer=tt.optim.Adam(lr=0.0005)
# )

In [ ]:
xgboost_features = ['prod_type_BM', 'graft_type', 'conditioning_intensity', 'prod_type_PB', 'cyto_score_detail', 'comorbidity_score', 'gvhd_proph_FK+ MMF +- others', 'dri_score', 'cyto_score', 'in_vivo_tcd', 'cardiac_Yes', 'sex_match', 'cmv_status', 'tbi_status_No TBI', 'karnofsky_score', 'psych_disturb', 'hla_high_res_8', 'race_group_Black or African-American', 'year_hct', 'age_at_hct', 'prim_disease_hct_IIS', 'tbi_status_TBI +- Other, <=cGy', 'prim_disease_hct_IEA', 'hla_match_drb1_high', 'hla_match_c_high', 'pulm_severe', 'prim_disease_hct_HIS', 'donor_related_Related', 'hla_match_a_high', 'cardiac_No', 'hla_match_b_low', 'gvhd_proph_Cyclophosphamide +- others', 'prior_tumor', 'tbi_status_TBI + Cy +- Other', 'hla_match_a_low', 'renal_issue', 'hepatic_severe', 'gvhd_proph_Other GVHD Prophylaxis', 'vent_hist', 'diabetes', 'hepatic_mild', 'race_group_White', 'hla_low_res_6', 'prim_disease_hct_SAA', 'donor_age', 'ethnicity_Non-resident of the U.S.', 'prim_disease_hct_Solid tumor', 'tce_imm_match_G/G', 'prim_disease_hct_IMD', 'tce_imm_match_Unknown', 'prim_disease_hct_ALL', 'gvhd_proph_CSA + MMF +- others(not FK)', 'ethnicity_Not Hispanic or Latino', 'tce_match', 'gvhd_proph_TDEPLETION alone', 'hla_match_dqb1_high', 'hla_match_c_low', 'tbi_status_TBI +- Other, >cGy', 'hla_nmdp_6', 'pulm_moderate_Unknown', 'prim_disease_hct_IPA', 'hla_high_res_10', 'peptic_ulcer', 'tce_imm_match_H/H', 'hla_match_b_high', 'race_group_American Indian or Alaska Native', 'prim_disease_hct_PCD', 'tce_div_match_GvH non-permissive', 'gvhd_proph_FKalone', 'gvhd_proph_TDEPLETION +- other', 'tce_div_match_Unknown', 'prim_disease_hct_HD', 'mrd_hct', 'race_group_More than one race', 'tce_imm_match_H/B', 'tce_imm_match_P/P', 'melphalan_dose_MEL', 'prim_disease_hct_NHL', 'arrhythmia', 'hla_match_dqb1_low', 'tce_div_match_Bi-directional non-permissive', 'race_group_Native Hawaiian or other Pacific Islander', 'tce_imm_match_P/H', 'gvhd_proph_Cyclophosphamide alone', 'donor_related_Multiple donor (non-UCB)', 'cardiac_Unknown', 'pulm_moderate_Yes', 'pulm_moderate_No', 'hla_high_res_6', 'hla_low_res_10', 'prim_disease_hct_MDS', 'donor_related_Unrelated', 'tbi_status_TBI +- Other, -cGy, fractionated', 'hla_low_res_8', 'tce_div_match_HvG non-permissive', 'obesity_Yes', 'tce_div_match_Permissive mismatched', 'prim_disease_hct_AI', 'rituximab', 'gvhd_proph_CDselect alone', 'prim_disease_hct_MPN', 'rheum_issue', 'obesity_Not done', 'tce_imm_match_G/B', 'prim_disease_hct_Other leukemia', 'ethnicity_Hispanic or Latino', 'hla_match_drb1_low', 'pulm_moderate_Not done', 'prim_disease_hct_AML', 'prim_disease_hct_Other acute leukemia', 'cardiac_Not done', 'gvhd_proph_No GvHD Prophylaxis', 'obesity_No', 'gvhd_proph_FK+ MTX +- others(not MMF)', 'gvhd_proph_CSA alone', 'gvhd_proph_Parent Q = yes, but no agent', 'melphalan_dose_N/A, Mel not given', 'prim_disease_hct_CML', 'ethnicity_Missing', 'tbi_status_TBI +- Other, -cGy, unknown dose', 'gvhd_proph_CSA + MTX +- others(not MMF,FK)', 'race_group_Asian']

In [ ]:
from sklearn.model_selection import KFold
from lifelines.utils import concordance_index as concordance_td
import warnings
warnings.filterwarnings("ignore")

# Use only the top 30 selected features
selected_features = xgboost_features[:30]

# Define number of folds
kf = KFold(n_splits=5, shuffle=True, random_state=42)
c_indices = []

# Loop through each fold
for fold, (train_idx, val_idx) in enumerate(kf.split(train_df)):
    print(f"Training Fold {fold + 1}/5...")

    # Split the dataset using only selected features
    train_fold, val_fold = train_df.iloc[train_idx], train_df.iloc[val_idx]

    event_col = "efs"      # Survival event (1 = event occurred, 0 = censored)
    time_col = "efs_time"

    # Convert boolean columns to int
    for col in train_fold.columns:
        if train_fold[col].dtype == "bool":
            train_fold[col] = train_fold[col].astype(int)
            val_fold[col] = val_fold[col].astype(int)

    # Convert to tensors
    train_x_fold = torch.tensor(train_fold[selected_features].values, dtype=torch.float32)
    val_x_fold = torch.tensor(val_fold[selected_features].values, dtype=torch.float32)

    train_durations_fold = torch.tensor(train_fold[time_col].values, dtype=torch.int64)
    train_events_fold = torch.tensor(train_fold[event_col].values, dtype=torch.int64)

    val_durations_fold = torch.tensor(val_fold[time_col].values, dtype=torch.int64)
    val_events_fold = torch.tensor(val_fold[event_col].values, dtype=torch.int64)

    # Define model for this fold using the same DeepSurv architecture
    input_dim = train_x_fold.shape[1]
    model = CoxPH(
        net=DeepSurvModel(input_dim)

    )

    optimizer = tt.optim.Adam(0.005, weight_decay=0.01)(model.net.parameters())
    # optimizer = torch.optim.RMSprop(model.net.parameters(), lr=0.001, alpha=0.9)


    # Train the model
    for epoch in range(1000):  # Adjust epochs if needed
        optimizer.zero_grad()
        risk_pred = model.net(train_x_fold).flatten()
        loss = CoxPHLoss()(risk_pred, train_durations_fold, train_events_fold)
        loss.backward()
        optimizer.step()

    # Evaluate on validation fold
    val_risk_pred = model.predict(val_x_fold).numpy().flatten()
    c_index_fold = concordance_td(
        val_durations_fold.numpy(), -val_risk_pred, val_events_fold.numpy()
    )
    c_indices.append(c_index_fold)

    print(f"Fold {fold + 1} C-Index: {c_index_fold:.4f}")

# Compute mean CV score
cv_c_index = np.mean(c_indices)
print(f"\nMean CV C-Index: {cv_c_index:.4f}")

Training Fold 1/5...
Fold 1 C-Index: 0.6575
Training Fold 2/5...
Fold 2 C-Index: 0.6585
Training Fold 3/5...
Fold 3 C-Index: 0.6614
Training Fold 4/5...
Fold 4 C-Index: 0.6528
Training Fold 5/5...
Fold 5 C-Index: 0.6429

Mean CV C-Index: 0.6546


In [ ]:
# Ensure test data is preprocessed similarly to train data
test_ids = test_df["ID"]  # Extract ID column before dropping it
# Convert boolean columns to int
for col in test_df.columns:
    if test_df[col].dtype == "bool":
        test_df[col] = test_df[col].astype(int)
test_x = torch.tensor(test_df[selected_features].values, dtype=torch.float32)  # Convert to tensor

# Make predictions on the test set
test_predictions = model.predict(test_x).numpy().flatten()

# Create output DataFrame with ID and Predictions
output_df = pd.DataFrame({"ID": test_ids, "prediction": test_predictions})

# Save to CSV for submission
output_df.to_csv("submission.csv", index=False)

# Display the first few rows of the output
print(output_df.head())